In [9]:
import pyvista as pv
import numpy as np
import asyncio

# 1. Setup State (Added x_dot for momentum)
state = {
    "x": 0.0,
    "x_prev": 0.0,
    "x_dot": 0.0,
    "theta": 0.3,
    "w": 0.0,
    "l": 2.0,
    "g": 9.81,
    "dt": 0.03,
    "running": True,
}

plotter = pv.Plotter(notebook=True, window_size=[600, 400])
plotter.set_background("black")

# 2. Geometry Setup
cart_mesh = pv.Cube(center=(state["x"], 0, 0), x_length=0.8, y_length=0.4, z_length=0.4)
cart_actor = plotter.add_mesh(cart_mesh, color="white")


def get_pole_points():
    p0 = np.array([state["x"], 0, 0])
    p1 = p0 + np.array(
        [state["l"] * np.sin(state["theta"]), -state["l"] * np.cos(state["theta"]), 0]
    )
    return p0, p1


p0, p1 = get_pole_points()
pole_actor = plotter.add_mesh(pv.Line(p0, p1), color="cyan", line_width=10)
plotter.add_mesh(
    pv.Plane(
        center=(0, -state["l"] - 0.2, 0), direction=(0, 1, 0), i_size=10, j_size=10
    ),
    color="gray",
    opacity=0.3,
)

plotter.view_xy()


# 3. INTERACTION: The Point Widget
def move_cart_callback(point):
    """This runs whenever you drag the magenta sphere"""
    state["x"] = point[0]  # Update the state x to match the handle
    update_scene()


# Add a magenta handle you can grab with your mouse
# Add the draggable handle (it's called a sphere widget in PyVista)
plotter.add_sphere_widget(
    move_cart_callback, center=(0, 0, 0), radius=0.2, color="magenta"
)


# 4. Updated Scene Logic
def update_scene():
    new_cart = pv.Cube(
        center=(state["x"], 0, 0), x_length=0.8, y_length=0.4, z_length=0.4
    )
    cart_actor.mapper.dataset.copy_from(new_cart)

    p0, p1 = get_pole_points()
    pole_actor.mapper.dataset.copy_from(pv.Line(p0, p1))
    plotter.render()


# 5. Physics with Momentum
async def sim_loop():
    while True:
        if state["running"]:
            # Calculate cart acceleration from mouse movement
            state["x_dot"] = (state["x"] - state["x_prev"]) / state["dt"]
            cart_accel = (state["x"] - state["x_prev"]) / (
                state["dt"] ** 2
            )  # Simplified for feel
            state["x_prev"] = state["x"]

            # Physics including the "push" from the cart
            accel = -(state["g"] / state["l"]) * np.sin(state["theta"]) - (
                cart_accel * np.cos(state["theta"]) / state["l"]
            )

            state["w"] += accel * state["dt"]
            state["w"] *= 0.98  # Damping
            state["theta"] += state["w"] * state["dt"]

            update_scene()
        await asyncio.sleep(state["dt"])


# 6. Run
plotter.show(jupyter_backend="trame")

loop = asyncio.get_event_loop()
loop.create_task(sim_loop())

Widget(value='<iframe src="http://localhost:49901/index.html?ui=P_0x1786d2210_7&reconnect=auto" class="pyvista…

<Task pending name='Task-419' coro=<sim_loop() running at /var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_59861/3077566491.py:74>>

In [10]:
import pyvista as pv
import numpy as np
import asyncio

# 1. State
state = {
    "x": 0.0,
    "theta": 0.3,
    "w": 0.0,
    "l": 2.0,
    "g": 9.81,
    "dt": 0.03,
    "running": True,
}

# 2. Plotter
plotter = pv.Plotter(notebook=True)
plotter.set_background("black")

# 3. Geometry
cart_actor = plotter.add_mesh(pv.Cube(center=(0, 0, 0)), color="white")
pole_actor = plotter.add_mesh(
    pv.Line([0, 0, 0], [0, -2, 0]), color="cyan", line_width=10
)
plotter.view_xy()


# 4. Interaction (The "Safe" Slider)
def move_cart(value):
    state["x"] = value
    # We update the visuals immediately for smoothness
    update_scene()


# This puts a slider INSIDE the 3D window
plotter.add_slider_widget(
    callback=move_cart,
    rng=[-4, 4],
    value=0,
    title="Cart X",
    pointa=(0.05, 0.1),  # Screen coordinates
    pointb=(0.35, 0.1),
    style="modern",
    color="magenta",
)


# 5. Scene Update
def update_scene():
    pivot = [state["x"], 0, 0]
    tip = [
        state["x"] + state["l"] * np.sin(state["theta"]),
        -state["l"] * np.cos(state["theta"]),
        0,
    ]

    # Update Meshes
    cart_actor.mapper.dataset.copy_from(
        pv.Cube(center=pivot, x_length=0.8, y_length=0.4, z_length=0.4)
    )
    pole_actor.mapper.dataset.copy_from(pv.Line(pivot, tip))
    plotter.render()


# 6. Physics Loop
async def sim_loop():
    while True:
        if state["running"]:
            accel = -(state["g"] / state["l"]) * np.sin(state["theta"])
            state["w"] += accel * state["dt"]
            state["w"] *= 0.98
            state["theta"] += state["w"] * state["dt"]
            update_scene()
        await asyncio.sleep(state["dt"])


# 7. Start
plotter.show(jupyter_backend="trame")

loop = asyncio.get_event_loop()
loop.create_task(sim_loop())

Widget(value='<iframe src="http://localhost:49901/index.html?ui=P_0x329e91e50_8&reconnect=auto" class="pyvista…

<Task pending name='Task-11554' coro=<sim_loop() running at /var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_59861/920282343.py:66>>

In [11]:
import pyvista as pv
import numpy as np
import asyncio

# 1. State & Constants
state = {
    "x": 0.0,
    "x_prev": 0.0,
    "x_velocity": 0.0,
    "theta": 0.3,
    "w": 0.0,
    "l": 2.0,
    "g": 9.81,
    "dt": 0.02,  # Higher frequency for better "feel"
}

# 2. Geometry Setup
plotter = pv.Plotter(notebook=True)
plotter.set_background("#050505")

cart_actor = plotter.add_mesh(
    pv.Cube(center=(0, 0, 0), x_length=0.8, y_length=0.4, z_length=0.4), color="white"
)
pole_actor = plotter.add_mesh(
    pv.Line([0, 0, 0], [0, -2, 0]), color="cyan", line_width=8
)

# Add a grid so you can see the movement clearly
plotter.add_mesh(
    pv.Plane(center=(0, -2.2, 0), i_size=20, j_size=20),
    color="white",
    opacity=0.1,
    show_edges=True,
)

plotter.view_xy()
plotter.camera.zoom(1.5)


# 3. Interactive Force Input
def on_slider_move(value):
    """
    This is called every time you move the slider.
    We update state['x'], and the physics loop calculates the force.
    """
    state["x"] = value


plotter.add_slider_widget(
    callback=on_slider_move,
    rng=[-5, 5],
    value=0,
    title="Drag to exert force",
    pointa=(0.6, 0.1),
    pointb=(0.9, 0.1),
    style="modern",
    color="magenta",
)


# 4. Scene Update Function
def update_scene():
    pivot = np.array([state["x"], 0, 0])
    tip = pivot + np.array(
        [state["l"] * np.sin(state["theta"]), -state["l"] * np.cos(state["theta"]), 0]
    )

    # Update GPU Buffers
    cart_actor.mapper.dataset.copy_from(
        pv.Cube(center=pivot, x_length=0.8, y_length=0.4, z_length=0.4)
    )
    pole_actor.mapper.dataset.copy_from(pv.Line(pivot, tip))
    plotter.render()


# 5. Physics Loop with Inertial Forces
async def physics_loop():
    while True:
        # --- CART DYNAMICS ---
        # Calculate velocity of the cart (how fast are you moving the slider?)
        current_v = (state["x"] - state["x_prev"]) / state["dt"]
        # Calculate acceleration (the 'jerk' of your hand)
        cart_accel = (current_v - state["x_velocity"]) / state["dt"]

        # Update trackers
        state["x_prev"] = state["x"]
        state["x_velocity"] = current_v

        # --- PENDULUM DYNAMICS ---
        # Equation: alpha = -(g/L)*sin(theta) - (accel/L)*cos(theta)
        # The (cart_accel) term is the force your hand exerts on the rod!
        alpha = -(state["g"] / state["l"]) * np.sin(state["theta"]) - (
            cart_accel / state["l"]
        ) * np.cos(state["theta"])

        state["w"] += alpha * state["dt"]
        state["w"] *= 0.97  # Damping/Air resistance
        state["theta"] += state["w"] * state["dt"]

        update_scene()
        await asyncio.sleep(state["dt"])


# 6. Show and Start
plotter.show(jupyter_backend="trame")

loop = asyncio.get_event_loop()
loop.create_task(physics_loop())

Widget(value='<iframe src="http://localhost:49901/index.html?ui=P_0x329e92850_9&reconnect=auto" class="pyvista…

<Task pending name='Task-51970' coro=<physics_loop() running at /var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_59861/1629971071.py:77>>

In [ ]:
import numpy as np
import asyncio
from bqplot import LinearScale, Scatter, Lines, Figure
import ipywidgets as widgets
from IPython.display import display

# 1. State
state = {
    "x_cart": 0.0,
    "x_prev": 0.0,
    "v_cart": 0.0,
    "theta": 0.5,
    "w": 0.0,
    "l": 1.0,
    "g": 9.81,
    "dt": 0.02,
}

# 2. Setup Plot
sc_x = LinearScale(min=-2.5, max=2.5)
sc_y = LinearScale(min=-1.5, max=1.5)

cart = Scatter(
    x=[state["x_cart"]],
    y=[0],
    scales={"x": sc_x, "y": sc_y},
    colors=["white"],
    marker="square",
    default_size=1000,
    enable_move=True,
    continuous_update=True,
)


def get_pole_coords():
    px = state["x_cart"] + state["l"] * np.sin(state["theta"])
    py = -state["l"] * np.cos(state["theta"])
    return [state["x_cart"], px], [0, py]


lx, ly = get_pole_coords()
pole = Lines(x=lx, y=ly, scales={"x": sc_x, "y": sc_y}, colors=["cyan"], stroke_width=4)
tip = Scatter(
    x=[lx[1]],
    y=[ly[1]],
    scales={"x": sc_x, "y": sc_y},
    colors=["magenta"],
    default_size=50,
)


# 3. Enhanced Redraw Function
def update_plot():
    lx, ly = get_pole_coords()
    # This context manager forces bqplot to update everything in ONE frame
    with fig.hold_sync():
        pole.x, pole.y = lx, ly
        cart.x = [state["x_cart"]]
        tip.x, tip.y = [lx[1]], [ly[1]]


# 4. Immediate Interaction (Fixes the "Wait for Drop" issue)
def on_cart_move(change):
    # Update the X position as fast as the mouse moves
    state["x_cart"] = change["new"][0]
    # FORCE a redraw right now, even while the mouse button is down
    update_plot()


cart.observe(on_cart_move, names=["x"])


# 5. Physics Loop
async def physics_loop():
    while True:
        # Calculate velocity/acceleration of the hand
        current_v = (state["x_cart"] - state["x_prev"]) / state["dt"]
        cart_accel = (current_v - state["v_cart"]) / state["dt"]

        state["x_prev"] = state["x_cart"]
        state["v_cart"] = current_v

        # Pendulum Math with Cart Acceleration Force
        alpha = -(state["g"] / state["l"]) * np.sin(state["theta"]) - (
            cart_accel / state["l"]
        ) * np.cos(state["theta"])

        state["w"] += alpha * state["dt"]
        state["w"] *= 0.97
        state["theta"] += state["w"] * state["dt"]

        update_plot()
        await asyncio.sleep(state["dt"])


# 6. Figure Setup
fig = Figure(
    marks=[pole, cart, tip],
    title="Physics active WHILE dragging",
    background_style={"fill": "#121212"},
    animation_duration=0,
)  # Must be 0 for real-time feel

display(fig)

loop = asyncio.get_event_loop()
loop.create_task(physics_loop())

Figure(background_style={'fill': '#1a1a1a'}, fig_margin={'top': 60, 'bottom': 60, 'left': 60, 'right': 60}, la…

<Task pending name='Task-79717' coro=<physics_loop() running at /var/folders/0r/235txxxd1p73wr0r5k9v6mqr0000gn/T/ipykernel_59861/405806431.py:72>>